# The goal of this notebook is to extract match ids data from different tiers + ranks to compose the initial match id dataset for the research
## Sub Goals:
        1) The goal is to use the Merged_Player_IDs_{...}.csv and extract Player IDs from it. Where we will need to extract 1 ID per tier and rank, then extract 50 match_ids.
        2) If that player does not have enought matches we will have to the next player in the same rank

## Result:
        1) lower_rank_player_ids_collected = (7 Ranks) * (4 divisions) * (50 solo queue matches) 
                == 40,000 matches across all tiers
                == 4,000 matches for each tier

# Importing the needed Libraries

In [1]:
from dotenv import load_dotenv
import os
import requests
import pandas as pd
import time
from datetime import datetime

load_dotenv()
api_key = os.environ.get('riot_api_key')

# Function that saves the IDs to CSV files Locally

# Rate Limiter + Utility Function + Main Function + Data Saver Function

In [2]:
class RateLimiter:
    '''Rate limiter for Riot Games API.
    
    Enforces Riot's rate limits:
    - 20 requests per 1 second (short-term bucket)
    - 100 requests per 2 minutes (long-term bucket)
    '''
    
    def __init__(self):
        self.short_term_requests = []
        self.long_term_requests = []
        self.short_window = 1
        self.long_window = 120
        self.short_limit = 20
        self.long_limit = 100
    
    def wait_if_needed(self):
        '''Check both rate limits and wait if necessary.'''
        current_time = time.time()
        
        self.short_term_requests = [t for t in self.short_term_requests 
                                     if current_time - t < self.short_window]
        self.long_term_requests = [t for t in self.long_term_requests 
                                    if current_time - t < self.long_window]
        
        if len(self.short_term_requests) >= self.short_limit:
            sleep_time = self.short_term_requests[0] + self.short_window - current_time
            if sleep_time > 0:
                print(f"Short-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
        
        if len(self.long_term_requests) >= self.long_limit:
            sleep_time = self.long_term_requests[0] + self.long_window - current_time
            if sleep_time > 0:
                print(f"Long-term rate limit reached. Waiting {sleep_time:.2f}s...")
                time.sleep(sleep_time)
                return self.wait_if_needed()
    
    def add_request(self):
        '''Record a new API request.'''
        current_time = time.time()
        self.short_term_requests.append(current_time)
        self.long_term_requests.append(current_time)


def get_match_history_ids(region='europe', sub_region='eun1', puuid=None, start=0, count=20, queue=420):
    """Retrieve a list of match IDs from a player's match history."""
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/by-puuid/{puuid}/ids'

    query_params = (
        f'?queue={queue}'
        f'&start={start}'
        f'&count={count}'
    )

    response = requests.get(
        root_url + endpoint + query_params + '&api_key=' + api_key
    )

    return response.json()


def collect_match_ids_from_players(api_key, region='europe', sub_region='eun1', matches_per_tier=None):
    """Collect match IDs per tier+division from player pool."""
    
    if matches_per_tier is None:
        matches_per_tier = {
            'IRON': 50, 'BRONZE': 50, 'SILVER': 50, 'GOLD': 50,
            'PLATINUM': 50, 'EMERALD': 50, 'DIAMOND': 50,
            'MASTER': 200, 'GRANDMASTER': 200, 'CHALLENGER': 200
        }
    elif isinstance(matches_per_tier, int):
        matches_per_tier = {tier: matches_per_tier for tier in 
                           ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']}
    
    limiter = RateLimiter()
    
    current_dir = os.getcwd()
    data_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'player ids'))
    
    all_files = [f for f in os.listdir(data_path) if f.startswith('Merged_Player_IDs_') and f.endswith('.csv')]
    if not all_files:
        print("No merged player file found")
        return {}
    
    latest_file = sorted(all_files, reverse=True)[0]
    df = pd.read_csv(os.path.join(data_path, latest_file))
    
    tier_order = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']
    div_order = ['I', 'II', 'III', 'IV']
    
    ordered_divisions = []
    for tier in tier_order:
        if tier in ['MASTER', 'GRANDMASTER', 'CHALLENGER']:
            ordered_divisions.append(f"{tier}_I")
        else:
            for div in div_order:
                ordered_divisions.append(f"{tier}_{div}")
    
    results = {}
    
    for division in ordered_divisions:
        tier = division.split('_')[0]
        rank = division.split('_')[1] if '_' in division else 'I'
        target_matches = matches_per_tier.get(tier, 50)
        
        div_players = df[df['division'] == division]
        
        if div_players.empty:
            print(f"\n{division}: No players found")
            continue
        
        print(f"\n{division} (target: {target_matches} matches):")
        match_ids = []
        player_index = 0
        
        while len(match_ids) < target_matches and player_index < len(div_players):
            player_row = div_players.iloc[player_index]
            puuid = player_row['puuid']
            queue_type = player_row['queueType'] if 'queueType' in player_row else 'RANKED_SOLO_5x5'
            
            start = 0
            while len(match_ids) < target_matches:
                batch_count = min(100, target_matches - len(match_ids))
                
                try:
                    limiter.wait_if_needed()
                    batch = get_match_history_ids(region=region, sub_region=sub_region, puuid=puuid, start=start, count=batch_count, queue=420)
                    limiter.add_request()
                    
                    if not batch:
                        print(f"  Player {player_index+1}: No more matches, switching player")
                        break
                    
                    match_ids.extend(batch)
                    start += len(batch)
                    
                except Exception as e:
                    print(f"  Error: {e}")
                    time.sleep(5)
                    break
            
            if len(match_ids) >= target_matches:
                print(f"  Got {len(match_ids)} matches from {player_index+1} player(s)")
                break
            else:
                print(f"  Player {player_index+1}: {len(match_ids)}/{target_matches} matches")
                player_index += 1
        
        # Store with metadata
        results[division] = {
            'tier': tier,
            'rank': rank,
            'queueType': queue_type,
            'region': region,
            'sub_region': sub_region,
            'match_ids': match_ids[:target_matches]
        }
        print(f"  Final: {len(results[division]['match_ids'])} matches")
    
    return results


def save_match_ids_to_csv(match_ids_dict, base_path=None): 
    """Save collected match IDs with tier, rank, queueType, region, sub_region columns."""
    
    if base_path is None:
        current_dir = os.getcwd()
        base_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'match ids'))
    
    os.makedirs(base_path, exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    for division, data in match_ids_dict.items():
        if not data['match_ids']:
            continue
        
        rows = []
        for match_id in data['match_ids']:
            rows.append({
                'tier': data['tier'],
                'rank': data['rank'],
                'match_id': match_id,
                'queueType': data['queueType'],
                'region': data['region'],
                'sub_region': data['sub_region']
            })
        
        df = pd.DataFrame(rows)
        filename = f"{division}_matches_{timestamp}.csv"
        filepath = os.path.join(base_path, filename)
        df.to_csv(filepath, index=False)
        print(f"Saved {len(df)} match IDs to: {filepath}")



In [3]:

match_ids = collect_match_ids_from_players(api_key, region='europe', sub_region='eun1', matches_per_tier={
    'IRON': 50, 'BRONZE': 50, 'SILVER': 50, 'GOLD': 50,
    'PLATINUM': 50, 'EMERALD': 50, 'DIAMOND': 50,
    'MASTER': 200, 'GRANDMASTER': 200, 'CHALLENGER': 200
})
save_match_ids_to_csv(match_ids)


IRON_I (target: 50 matches):
  Got 50 matches from 1 player(s)
  Final: 50 matches

IRON_II (target: 50 matches):
  Player 1: No more matches, switching player
  Player 1: 34/50 matches
  Got 50 matches from 2 player(s)
  Final: 50 matches

IRON_III (target: 50 matches):
  Got 50 matches from 1 player(s)
  Final: 50 matches

IRON_IV (target: 50 matches):
  Got 50 matches from 1 player(s)
  Final: 50 matches

BRONZE_I (target: 50 matches):
  Got 50 matches from 1 player(s)
  Final: 50 matches

BRONZE_II (target: 50 matches):
  Player 1: No more matches, switching player
  Player 1: 27/50 matches
  Got 50 matches from 2 player(s)
  Final: 50 matches

BRONZE_III (target: 50 matches):
  Player 1: No more matches, switching player
  Player 1: 14/50 matches
  Got 50 matches from 2 player(s)
  Final: 50 matches

BRONZE_IV (target: 50 matches):
  Got 50 matches from 1 player(s)
  Final: 50 matches

SILVER_I (target: 50 matches):
  Got 50 matches from 1 player(s)
  Final: 50 matches

SILVER_II

# Merging/Combining all the CSV files together 

In [4]:
def merge_match_id_files(base_path=None):
    """Merge all division match ID CSV files into a single CSV maintaining rank order."""
    
    if base_path is None:
        current_dir = os.getcwd()
        base_path = os.path.abspath(os.path.join(current_dir, '..', 'data', 'data collected', 'match ids'))
    
    # Define order: Iron to Challenger
    tier_order = ['IRON', 'BRONZE', 'SILVER', 'GOLD', 'PLATINUM', 'EMERALD', 'DIAMOND', 'MASTER', 'GRANDMASTER', 'CHALLENGER']
    div_order = ['I', 'II', 'III', 'IV']
    
    # Build ordered division list
    ordered_divisions = []
    for tier in tier_order:
        if tier in ['MASTER', 'GRANDMASTER', 'CHALLENGER']:
            ordered_divisions.append(f"{tier}_I")
        else:
            for div in div_order:
                ordered_divisions.append(f"{tier}_{div}")
    
    # Find latest file for each division
    all_files = os.listdir(base_path)
    division_files = {}
    
    for div in ordered_divisions:
        matching = [f for f in all_files if f.startswith(div) and f.endswith('.csv')]
        if matching:
            # Get most recent file (last timestamp)
            matching.sort(reverse=True)
            division_files[div] = os.path.join(base_path, matching[0])
    
    # Read and merge
    dfs = []
    for div in ordered_divisions:
        if div in division_files:
            df = pd.read_csv(division_files[div])
            df['division'] = div
            dfs.append(df)
            print(f"Loaded {div}: {len(df)} match IDs")
    
    merged = pd.concat(dfs, ignore_index=True)
    
    # Save merged file
    from datetime import datetime
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_path = os.path.join(base_path, f"Merged_Match_IDs_{timestamp}.csv")
    merged.to_csv(output_path, index=False)
    
    print(f"\nMerged {len(merged)} match IDs to: {output_path}")
    return merged

# Usage
all_matches = merge_match_id_files()

Loaded IRON_I: 50 match IDs
Loaded IRON_II: 50 match IDs
Loaded IRON_III: 50 match IDs
Loaded IRON_IV: 50 match IDs
Loaded BRONZE_I: 50 match IDs
Loaded BRONZE_II: 50 match IDs
Loaded BRONZE_III: 50 match IDs
Loaded BRONZE_IV: 50 match IDs
Loaded SILVER_I: 50 match IDs
Loaded SILVER_II: 50 match IDs
Loaded SILVER_III: 50 match IDs
Loaded SILVER_IV: 50 match IDs
Loaded GOLD_I: 50 match IDs
Loaded GOLD_II: 50 match IDs
Loaded GOLD_III: 50 match IDs
Loaded GOLD_IV: 50 match IDs
Loaded PLATINUM_I: 50 match IDs
Loaded PLATINUM_II: 50 match IDs
Loaded PLATINUM_III: 50 match IDs
Loaded PLATINUM_IV: 50 match IDs
Loaded EMERALD_I: 50 match IDs
Loaded EMERALD_II: 50 match IDs
Loaded EMERALD_III: 50 match IDs
Loaded EMERALD_IV: 50 match IDs
Loaded DIAMOND_I: 50 match IDs
Loaded DIAMOND_II: 50 match IDs
Loaded DIAMOND_III: 50 match IDs
Loaded DIAMOND_IV: 50 match IDs
Loaded MASTER_I: 200 match IDs
Loaded GRANDMASTER_I: 200 match IDs
Loaded CHALLENGER_I: 200 match IDs

Merged 2000 match IDs to: c:\